# 面试题：学习型 Tool Router 怎样训练和兜底？

回答要点：Router 将请求映射到 no-tool、工具簇或具体工具；训练数据来自已完成且版本化的调用轨迹。它只做候选选择，不能绕过 allowlist、schema 和权限。低置信度、并列候选和不可填参数必须回退到追问、检索或人工。评估同时看 top-k 召回、可执行率、错误副作用、成本和延迟。

## 真实案例

售后助手在物流、退款、地址变更与 no-tool 之间路由。

## 基线

基线选择工具目录里的第一个关键词命中项。

## 结果解读

Notebook 手写可解释的词项权重学习，输出每类分数、置信度和兜底。

## 失败案例

“订单状态”没有足够线索，强选具体工具会产生错误副作用。

In [1]:
train = [('查物流进度','物流'), ('退款多久到账','退款'), ('修改收货地址','地址'), ('写商品卖点','no_tool'), ('物流单号在哪里','物流'), ('退款失败怎么办','退款')]  # 构造六条来自成功工单的脱敏训练轨迹。
test = [('订单物流到哪了','物流'), ('订单状态','fallback'), ('退回的钱什么时候到','退款'), ('换一个收货地址','地址'), ('生成促销文案','no_tool'), ('物流和退款都要查','fallback')]  # 构造六条包含明确与歧义请求的评测事件。
print('训练轨迹:', train)  # 输出 Router 学习所依据的历史事件。
print('评测请求:', [item[0] for item in test])  # 输出具有业务语义的待路由请求。

训练轨迹: [('查物流进度', '物流'), ('退款多久到账', '退款'), ('修改收货地址', '地址'), ('写商品卖点', 'no_tool'), ('物流单号在哪里', '物流'), ('退款失败怎么办', '退款')]
评测请求: ['订单物流到哪了', '订单状态', '退回的钱什么时候到', '换一个收货地址', '生成促销文案', '物流和退款都要查']


In [2]:
catalog = ['物流','退款','地址']  # 定义可执行工具的有序目录。
def first_match(text):  # 定义只选第一个命中词的脆弱基线。
    return next((tool for tool in catalog if tool in text), '物流')  # 没命中时错误地默认选择物流工具。
baseline = [(text, first_match(text)) for text, _ in test]  # 对六条评测请求计算 baseline 路由。
print('第一个匹配基线:', baseline)  # 输出 baseline 对歧义与无工具请求的错误倾向。

第一个匹配基线: [('订单物流到哪了', '物流'), ('订单状态', '物流'), ('退回的钱什么时候到', '物流'), ('换一个收货地址', '地址'), ('生成促销文案', '物流'), ('物流和退款都要查', '物流')]


In [3]:
weights = {}  # 初始化从词项到工具类别的可解释权重表。
for text, label in train:  # 遍历成功的历史调用轨迹。
    for token in text:  # 以字符作为无外部依赖的教学词项切分。
        weights[(token, label)] = weights.get((token, label), 0) + 1  # 为观测到的词项和工具标签累计权重。
def route(text):  # 定义手写打分、置信度和兜底路由器。
    scores = {label:sum(weights.get((token, label), 0) for token in text) for label in ['物流','退款','地址','no_tool']}  # 对每个类别累加文本词项权重。
    ordered = sorted(scores.items(), key=lambda item:item[1], reverse=True)  # 按分数排序得到最佳与次佳候选。
    margin = ordered[0][1] - ordered[1][1]  # 计算最佳候选相对次佳候选的可解释间隔。
    choice = ordered[0][0] if ordered[0][1] > 0 and margin >= 2 else 'fallback'  # 要求至少两个权重差，低证据或并列时回退而非强选工具。
    return choice, scores, margin  # 返回路由结果、全部中间分数和置信间隔。

In [4]:
results = [(text, expected) + route(text) for text, expected in test]  # 对六条评测请求执行训练后的 Router。
print('请求 | 期望 | Router | 分数 | 间隔')  # 输出包含每类分数的结果表标题。
for text, expected, choice, scores, margin in results:  # 遍历 Router 的每条可解释预测。
    print(text, expected, choice, scores, margin)  # 输出请求、标签、路由和中间量。
accuracy = sum(choice == expected for _, expected, choice, _, _ in results) / len(results)  # 计算严格包含 fallback 的教学集准确率。
print('Router 准确率:', round(accuracy, 2), '；这里验证机制，不代表线上模型泛化。')  # 输出受控教学实验指标与边界。

请求 | 期望 | Router | 分数 | 间隔
订单物流到哪了 物流 物流 {'物流': 6, '退款': 1, '地址': 0, 'no_tool': 0} 5
订单状态 fallback fallback {'物流': 1, '退款': 0, '地址': 0, 'no_tool': 0} 1
退回的钱什么时候到 退款 退款 {'物流': 0, '退款': 4, '地址': 0, 'no_tool': 0} 4
换一个收货地址 地址 地址 {'物流': 0, '退款': 0, '地址': 4, 'no_tool': 0} 4
生成促销文案 no_tool fallback {'物流': 0, '退款': 0, '地址': 0, 'no_tool': 0} 0
物流和退款都要查 fallback fallback {'物流': 5, '退款': 4, '地址': 0, 'no_tool': 0} 1
Router 准确率: 0.83 ；这里验证机制，不代表线上模型泛化。


In [5]:
bad_choice = '物流'  # 模拟把“订单状态”默认硬路由到物流的错误策略。
fixed_choice, fixed_scores, fixed_margin = route('订单状态')  # 使用间隔门槛得到可审计的兜底结果。
print('失败案例：默认=', bad_choice, '，Router=', fixed_choice, '，分数=', fixed_scores, '，间隔=', fixed_margin)  # 展示低置信度不应执行工具。
print('生产差距：真实 Router 需使用版本化轨迹、权限过滤、参数可填率、影子流量和按工具版本的回滚。')  # 说明词项教学实现与生产系统的差距。

失败案例：默认= 物流 ，Router= fallback ，分数= {'物流': 1, '退款': 0, '地址': 0, 'no_tool': 0} ，间隔= 1
生产差距：真实 Router 需使用版本化轨迹、权限过滤、参数可填率、影子流量和按工具版本的回滚。


In [6]:
assert fixed_choice == 'fallback'  # 验证无足够证据时 Router 不会强制选择工具。
assert route('退回的钱什么时候到')[0] == '退款'  # 验证退款语义会被正确路由。
assert len(weights) > 0  # 验证训练轨迹实际生成了可用权重。